# Deep Sets Issue #120 Benchmark and New Arms

This notebook is the primary Issue #120 artifact. It summarizes existing benchmark metrics, documents exact feature columns by config, and prepares deferred Slurm commands for three new feature regimes.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from build_deepsets_dataset import deepsets_point_feature_names

REPO_ROOT = Path.cwd()
FIG_DIR = REPO_ROOT / "analysis" / "figures" / "deepsets_issue120"
FIG_DIR.mkdir(parents=True, exist_ok=True)

EXISTING_ARMS = [
    {
        "label": "Baseline (curvature only)",
        "config": "configs/deepsets_ispy2_pointfeat_baseline.yaml",
        "regime": "baseline",
        "metrics_path": "experiments/deepsets_ispy2_pointfeat_baseline/train/deepsets_ispy2_pointfeat_baseline_20260421_125620/deepsets_ispy2_pointfeat_baseline/metrics.json",
    },
    {
        "label": "Geometry + topology",
        "config": "configs/deepsets_ispy2_pointfeat_geom_topo.yaml",
        "regime": "geometry_topology",
        "metrics_path": "experiments/deepsets_ispy2_pointfeat_geom_topo/train/deepsets_ispy2_pointfeat_geom_topo_20260421_130516/deepsets_ispy2_pointfeat_geom_topo/metrics.json",
    },
    {
        "label": "Geometry + topology + dynamic",
        "config": "configs/deepsets_ispy2_pointfeat_geom_topo_dynamic.yaml",
        "regime": "geometry_topology_dynamic",
        "metrics_path": "experiments/deepsets_ispy2_pointfeat_geom_topo_dynamic_existing_manifest_train/deepsets_ispy2_pointfeat_geom_topo_dynamic_20260504_184523/deepsets_ispy2_pointfeat_geom_topo_dynamic/metrics.json",
    },
]

NEW_ARMS = [
    {
        "label": "Geometry + topology + curvature",
        "config": "configs/deepsets_ispy2_pointfeat_geom_topo_plus_curvature.yaml",
        "regime": "geometry_topology_plus_curvature",
        "out_root": "experiments/deepsets_ispy2_pointfeat_geom_topo_plus_curvature",
    },
    {
        "label": "Geometry + topology (no shells)",
        "config": "configs/deepsets_ispy2_pointfeat_geom_topo_no_shells.yaml",
        "regime": "geometry_topology_no_shells",
        "out_root": "experiments/deepsets_ispy2_pointfeat_geom_topo_no_shells",
    },
    {
        "label": "Curvature + dynamic",
        "config": "configs/deepsets_ispy2_pointfeat_curvature_plus_dynamic.yaml",
        "regime": "curvature_plus_dynamic",
        "out_root": "experiments/deepsets_ispy2_pointfeat_curvature_plus_dynamic",
    },
]

## Existing benchmark metrics (with path checks)

In [ ]:
rows = []
missing_metrics = []
for arm in EXISTING_ARMS:
    metrics_file = REPO_ROOT / arm["metrics_path"]
    if not metrics_file.exists():
        missing_metrics.append(str(metrics_file))
        continue
    payload = json.loads(metrics_file.read_text(encoding="utf-8"))
    auc = float(payload["aggregated_metrics"]["auc"]["mean"])
    auc_std = float(payload["aggregated_metrics"]["auc"]["std"])
    n_features = len(deepsets_point_feature_names(arm["regime"]))
    rows.append(
        {
            "arm": arm["label"],
            "config": arm["config"],
            "regime": arm["regime"],
            "n_features": n_features,
            "auc_mean": auc,
            "auc_std": auc_std,
            "metrics_path": str(metrics_file),
        }
    )

if missing_metrics:
    print("Missing metrics paths:")
    for p in missing_metrics:
        print(" -", p)

existing_metrics_df = pd.DataFrame(rows)
existing_metrics_df

In [ ]:
if not existing_metrics_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.bar(
        existing_metrics_df["arm"],
        existing_metrics_df["auc_mean"],
        yerr=existing_metrics_df["auc_std"],
        capsize=6,
    )
    ax.set_ylabel("Validation AUC (mean ± std)")
    ax.set_ylim(0.45, 0.60)
    ax.set_title("Deep Sets feature-regime benchmark (existing arms)")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=12, ha="right")
    fig.tight_layout()
    out_png = FIG_DIR / "feature_set_auc_comparison.png"
    fig.savefig(out_png, dpi=180)
    plt.close(fig)
    print(f"Saved: {out_png}")
else:
    print("No existing metrics rows loaded; skipped benchmark plot.")

## Explicit feature lists by config

In [ ]:
feature_rows = []
for arm in EXISTING_ARMS + NEW_ARMS:
    names = deepsets_point_feature_names(arm["regime"])
    for i, feat in enumerate(names, start=1):
        feature_rows.append(
            {
                "config": arm["config"],
                "arm": arm["label"],
                "regime": arm["regime"],
                "feature_index": i,
                "feature_name": feat,
            }
        )

feature_table_df = pd.DataFrame(feature_rows)
feature_table_df

In [ ]:
for arm in EXISTING_ARMS + NEW_ARMS:
    display_df = feature_table_df.loc[
        feature_table_df["config"] == arm["config"], ["feature_index", "feature_name"]
    ]
    print(f"\n{arm['label']} :: {arm['config']} ({len(display_df)} features)")
    display(display_df.reset_index(drop=True))

## New Issue #120 configs (prepared only; execution deferred)

In [ ]:
deferred_commands = []
for arm in NEW_ARMS:
    cmd = (
        f"CONFIG={arm['config']} "
        f"OUT_ROOT={arm['out_root']} "
        "BUILD_SHARDS=8 PARTITION=general "
        "bash slurm/submit_deepsets_pipeline.sh"
    )
    deferred_commands.append(
        {
            "arm": arm["label"],
            "config": arm["config"],
            "deferred_slurm_command": cmd,
        }
    )

pd.DataFrame(deferred_commands)

**Deferred execution note:** The commands above are exact launch commands for the three new feature-regime arms. They are intentionally not executed in this workstream.

In [ ]:
if not existing_metrics_df.empty:
    summary_csv = FIG_DIR / "feature_set_benchmark_summary.csv"
    existing_metrics_df.to_csv(summary_csv, index=False)
    feature_csv = FIG_DIR / "feature_columns_by_config.csv"
    feature_table_df.to_csv(feature_csv, index=False)
    print(f"Saved: {summary_csv}")
    print(f"Saved: {feature_csv}")
else:
    print("Metrics summary CSV skipped because existing metrics were unavailable.")